# 冷冻周期临床信息整合 CHAID 分析（活产预测）

整合以下临床信息进行CHAID决策树分析，预测活产：
- QCP90（胚胎质量评分）
- Age（年龄）
- BMI
- 转化日内膜厚度
- 转化日孕酮
- 女性原发/继发
- 对应新鲜周期治疗方法

**目标变量：活产（胎儿数 > 0）**

In [368]:
import pandas as pd
import numpy as np
from CHAID import Tree
import matplotlib.pyplot as plt
from graphviz import Digraph
import warnings
import os

# 添加Graphviz到PATH（macOS Homebrew路径）
os.environ["PATH"] += os.pathsep + '/opt/homebrew/bin'

warnings.filterwarnings('ignore')

print("✅ 库导入成功")
print(f"PATH已更新，包含Graphviz路径")

✅ 库导入成功
PATH已更新，包含Graphviz路径


## 1. 数据读取

In [369]:
# 读取冷冻周期数据（2023-2025）
df_2023 = pd.read_excel("/Users/lzz/Documents/GitHub/IVF_AI/embryo_info/23.1.1-25.7.31time-lapse原始数据新鲜+冷冻/冷冻/2023-冷冻.xls")
df_2024 = pd.read_excel("/Users/lzz/Documents/GitHub/IVF_AI/embryo_info/23.1.1-25.7.31time-lapse原始数据新鲜+冷冻/冷冻/2024-冷冻.xls")
df_2025 = pd.read_excel("/Users/lzz/Documents/GitHub/IVF_AI/embryo_info/23.1.1-25.7.31time-lapse原始数据新鲜+冷冻/冷冻/2025-冷冻.xls")

# 合并数据
df_frozen = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

print(f"📊 冷冻周期数据总数: {len(df_frozen)} 条")
print(f"数据形状: {df_frozen.shape}")

📊 冷冻周期数据总数: 3723 条
数据形状: (3723, 88)


In [370]:
# 读取PC1数据（QCP90）
pc1_path = "/Users/lzz/Documents/GitHub/IVF_AI/prediction/code/STORK_lzz_torch/outputs/pc1_cycle_vgg16/pc1_cycle_level.csv"
df_pc1 = pd.read_csv(pc1_path)

print(f"📊 PC1数据: {len(df_pc1)} 条")
print(f"PC1列名: {df_pc1.columns.tolist()}")

📊 PC1数据: 1423 条
PC1列名: ['patient_id', 'pc1_median', 'pc1_p90', 'pc1_mean', 'pc1_std', 'pc1_max', 'n_embryos']


## 2. 数据预处理和特征提取

In [371]:
# 提取关键临床信息列（使用正确的列名）
clinical_cols = [
    '女病人ID', 'ARTID', '移植时年龄', 'Bmi(新鲜周期)', 
    '转化日内膜厚度', '转化日孕酮', '女性原发、继发', '移植数',
    '对应新鲜周期治疗方法', '胎儿数'
]

# 提取数据
df_clinical = df_frozen[clinical_cols].copy()

print(f"📊 临床数据形状: {df_clinical.shape}")
print(f"\\n前5行数据:")
print(df_clinical.head())

📊 临床数据形状: (3723, 10)
\n前5行数据:
    女病人ID   ARTID  移植时年龄  Bmi(新鲜周期)  转化日内膜厚度  转化日孕酮 女性原发、继发  移植数 对应新鲜周期治疗方法  \
0  286404  260123   27.2      40.06      9.7   1.63   继发性不孕    1        IVF   
1  138598  247840   31.5      36.21     11.0   0.97   继发性不孕    1        IVF   
2  138598  268488   31.9      36.21     15.0   0.81   继发性不孕    1        IVF   
3  285001  255700   25.9      36.05      9.5   0.77   继发性不孕    1        IVF   
4  267725  242884   31.5      35.16      9.5   1.65   继发性不孕    2        IVF   

   胎儿数  
0    0  
1    0  
2    1  
3    1  
4    0  


In [372]:
# 创建活产列（胎儿数 > 0）
df_clinical['活产'] = (df_clinical['胎儿数'] > 0).astype(int)

print(f"活产率: {df_clinical['活产'].mean():.2%}")
print(f"活产数: {df_clinical['活产'].sum()}")
print(f"非活产数: {(df_clinical['活产']==0).sum()}")

活产率: 34.81%
活产数: 1296
非活产数: 2427


In [373]:
# 合并PC1数据（注意：PC1文件用patient_id，冷冻数据用女病人ID）
df_clinical['patient_id'] = df_clinical['女病人ID']
df_merged = df_clinical.merge(df_pc1, on='patient_id', how='inner')

print(f"📊 合并后数据形状: {df_merged.shape}")
print(f"合并后活产率: {df_merged['活产'].mean():.2%}")

📊 合并后数据形状: (2305, 18)
合并后活产率: 37.70%


In [374]:
# 数据清洗：删除缺失值
print("📌 缺失值统计:")
print(df_merged.isnull().sum())

# 删除关键列的缺失值
df_clean = df_merged.dropna(subset=['移植时年龄', 'Bmi(新鲜周期)', '活产']).copy()

print(f"\n✅ 清洗后数据形状: {df_clean.shape}")
print(f"清洗后活产率: {df_clean['活产'].mean():.2%}")

📌 缺失值统计:
女病人ID          0
ARTID          0
移植时年龄          0
Bmi(新鲜周期)     10
转化日内膜厚度       11
转化日孕酮         56
女性原发、继发        1
移植数            0
对应新鲜周期治疗方法     0
胎儿数            0
活产             0
patient_id     0
pc1_median     0
pc1_p90        0
pc1_mean       0
pc1_std       68
pc1_max        0
n_embryos      0
dtype: int64

✅ 清洗后数据形状: (2295, 18)
清洗后活产率: 37.82%


In [375]:
# 只保留单胚胎移植的周期（移植数==1）
print(f"筛选前数据量: {len(df_clean)}")
df_clean = df_clean[df_clean['移植数'] == 1].copy()
print(f"筛选后数据量（仅单胚胎移植）: {len(df_clean)}")
print(f"单胚胎移植活产率: {df_clean['活产'].mean():.2%}")

筛选前数据量: 2295
筛选后数据量（仅单胚胎移植）: 1884
单胚胎移植活产率: 41.77%


## 3. 特征分箱处理

In [ ]:
# 年龄分箱（英文标签）
df_clean['Age_Group'] = pd.cut(
    df_clean['移植时年龄'], 
    bins=[0, 30, 35, 40, 100], 
    labels=['≤30', '31-35', '36-40', '>40']
)

# BMI分箱（英文标签，合并超重和肥胖）
df_clean['BMI_Group'] = pd.cut(
    df_clean['Bmi(新鲜周期)'], 
    bins=[0, 18.5, 24, 100], 
    labels=['Underweight', 'Normal', 'Overweight']
)

print("✅ Age and BMI binning completed")

In [377]:
# PC1（QCP）十分位分箱
df_clean['QCP_Decile'] = pd.qcut(
    df_clean['pc1_p90'], 
    q=10, 
    labels=['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10'],
    duplicates='drop'
)

print(f"✅ QCP十分位分箱完成")
print(f"\n各十分位分布:")
print(df_clean['QCP_Decile'].value_counts().sort_index())

✅ QCP十分位分箱完成

各十分位分布:
QCP_Decile
D1     189
D2     189
D3     187
D4     192
D5     185
D6     188
D7     189
D8     188
D9     188
D10    189
Name: count, dtype: int64


In [ ]:
# 转化日内膜厚度分箱（英文标签）
df_clean['内膜厚度_Group'] = pd.cut(
    df_clean['转化日内膜厚度'], 
    bins=[0, 8, 12, 100], 
    labels=['Thin', 'Normal', 'Thick']
)

# 转化日孕酮分箱（英文标签）
df_clean['孕酮_Group'] = pd.cut(
    df_clean['转化日孕酮'], 
    bins=[0, 1, 2, 100], 
    labels=['Low', 'Normal', 'High']
)

print("✅ Endometrial thickness and progesterone binning completed")

## 4. 准备CHAID分析数据

In [379]:
# 选择CHAID分析的特征（恢复完整特征列表）
chaid_features = [
    'Age_Group', 'BMI_Group', 'QCP_Decile', 
    '内膜厚度_Group', '孕酮_Group',
    '女性原发、继发', '对应新鲜周期治疗方法'
]

# 删除任何特征列的缺失值
df_chaid = df_clean.dropna(subset=chaid_features + ['活产']).copy()

print(f"📊 CHAID分析数据形状: {df_chaid.shape}")
print(f"活产率: {df_chaid['活产'].mean():.2%}")
print(f"特征数量: {len(chaid_features)}")

📊 CHAID分析数据形状: (1835, 23)
活产率: 41.74%
特征数量: 7


In [380]:
# 准备CHAID分析的完整DataFrame（包含特征和目标变量）
df_chaid_full = df_chaid[chaid_features + ['活产']].copy()

# 确保所有特征都是分类变量
for col in chaid_features:
    df_chaid_full[col] = df_chaid_full[col].astype(str)

print(f"✅ CHAID完整数据形状: {df_chaid_full.shape}")
print(f"✅ 列名: {df_chaid_full.columns.tolist()}")

✅ CHAID完整数据形状: (1835, 8)
✅ 列名: ['Age_Group', 'BMI_Group', 'QCP_Decile', '内膜厚度_Group', '孕酮_Group', '女性原发、继发', '对应新鲜周期治疗方法', '活产']


In [ ]:
# 构建CHAID决策树（使用QCP十分位，恢复完整特征）
i_variables = {
    'Age_Group': 'nominal',
    'BMI_Group': 'nominal',
    'QCP_Decile': 'nominal',
    '内膜厚度_Group': 'nominal',
    '孕酮_Group': 'nominal',
    '女性原发、继发': 'nominal',
    '对应新鲜周期治疗方法': 'nominal'
}

tree = Tree.from_pandas_df(
    df_chaid_full, 
    i_variables, 
    '活产',
    min_parent_node_size=200,
    max_depth=5,
    alpha_merge=0.05
)

print("✅ CHAID决策树构建完成")
print("\n📊 CHAID决策树结构：")
print(tree.print_tree())

✅ CHAID决策树构建完成

📊 CHAID决策树结构：


([], {0: 1069.0, 1: 766.0}, (Age_Group, p=3.9764644107421795e-17, score=79.47568169230514, groups=[['31-35'], ['36-40'], ['>40'], ['≤30']]), dof=3))
|-- (['31-35'], {0: 450.0, 1: 346.0}, (QCP_Decile, p=3.751350856841908e-10, score=39.238166391556994, groups=[['D1', 'D2', 'D3', 'D4', 'D5'], ['D10', 'D9', 'D6', 'D7', 'D8']]), dof=1))
|   |-- (['D1', 'D2', 'D3', 'D4', 'D5'], {0: 262.0, 1: 124.0}, (内膜厚度_Group, p=0.005268026754092169, score=7.785052237629726, groups=[['8-12', '>12'], ['<8']]), dof=1))
|   |   |-- (['8-12', '>12'], {0: 212.0, 1: 114.0}, (BMI_Group, p=0.006485083173528568, score=7.4103187725038175, groups=[['18.5-24', '>28'], ['24-28', '<18.5']]), dof=1))
|   |   |   |-- (['18.5-24', '>28'], {0: 124.0, 1: 84.0}, (孕酮_Group, p=0.03890103128858455, score=4.265184209884671, groups=[['1-2', '>2'], ['<1']]), dof=1))
|   |   |   |   |-- (['1-2', '>2'], {0: 47.0, 1: 44.0}, <Invalid Chaid Split> - the minimum parent node size threshold has been reached)
|   |   |   |   +-- (['<1'], {0

## 6. 叶节点活产率分析

In [382]:
# 提取叶节点信息（修正：使用与可视化相同的方法）
# 先构建节点映射和父子关系
node_map_leaf = {n.node_id: n for n in tree.tree_store}
children_leaf = {}
for n in tree.tree_store:
    if n.parent is not None:
        children_leaf.setdefault(n.parent, []).append(n.node_id)

def get_leaf_nodes(node_id, path="", leaves=None):
    if leaves is None:
        leaves = []
    
    node = node_map_leaf[node_id]
    is_leaf = not node.split.valid()
    
    if is_leaf:
        live_birth_rate = node.members[1] / len(node.members) if len(node.members) > 0 else 0
        leaves.append({
            'path': path,
            'n': len(node.members),
            'live_birth': node.members[1],
            'rate': live_birth_rate
        })
    else:
        for child_id in children_leaf.get(node_id, []):
            child = node_map_leaf[child_id]
            child_path = f"{path} -> {child.choices}" if path else str(child.choices)
            get_leaf_nodes(child_id, child_path, leaves)
    return leaves

leaves = get_leaf_nodes(tree.tree_store[0].node_id)
df_leaves = pd.DataFrame(leaves).sort_values('rate', ascending=False)

print("📌 叶节点活产率排序：")
print(df_leaves)

📌 叶节点活产率排序：
                                                 path  n  live_birth  rate
4   ['31-35'] -> ['D10', 'D9', 'D6', 'D7', 'D8'] -...  2       170.0  85.0
12  ['≤30'] -> ['D10', 'D9', 'D8', 'D2', 'D5', 'D6...  2       133.0  66.5
8   ['36-40'] -> ['D10', 'D4', 'D3', 'D7', 'D8', '...  2        94.0  47.0
5   ['31-35'] -> ['D10', 'D9', 'D6', 'D7', 'D8'] -...  2        52.0  26.0
11   ['≤30'] -> ['D1', 'D4', 'D3', 'D7'] -> ['继发性不孕']  2        52.0  26.0
13  ['≤30'] -> ['D10', 'D9', 'D8', 'D2', 'D5', 'D6...  2        51.0  25.5
0   ['31-35'] -> ['D1', 'D2', 'D3', 'D4', 'D5'] ->...  2        44.0  22.0
1   ['31-35'] -> ['D1', 'D2', 'D3', 'D4', 'D5'] ->...  2        40.0  20.0
10   ['≤30'] -> ['D1', 'D4', 'D3', 'D7'] -> ['原发性不孕']  2        36.0  18.0
7   ['36-40'] -> ['D10', 'D4', 'D3', 'D7', 'D8', '...  2        31.0  15.5
2   ['31-35'] -> ['D1', 'D2', 'D3', 'D4', 'D5'] ->...  2        30.0  15.0
9                                             ['>40']  2        13.0   6.5
3   ['31-35']

## 7. CHAID决策树可视化

In [ ]:
from graphviz import Digraph

# 变量名中英文映射
VAR_NAME_MAP = {
    'Age_Group': 'Age',
    'BMI_Group': 'BMI',
    'QCP_Decile': 'QCP',
    '内膜厚度_Group': 'Endometrial Thickness',
    '孕酮_Group': 'Progesterone',
    '女性原发、继发': 'Infertility Type',
    '对应新鲜周期治疗方法': 'Treatment Method'
}

# 分类值中英文映射
VALUE_MAP = {
    '原发性不孕': 'Primary',
    '继发性不孕': 'Secondary'
}

# 年龄分组合并显示
AGE_MERGE = {
    frozenset(["≤30"]): "≤30",
    frozenset(["31-35"]): "31-35",
    frozenset(["36-40"]): "36-40",
    frozenset([">40"]): ">40",
    frozenset(["≤30", "31-35"]): "≤35",
    frozenset(["36-40", ">40"]): "≥36"
}

def merge_age_label(choices):
    """合并年龄标签用于展示"""
    if not isinstance(choices, (list, tuple, set)):
        return choices
    s = frozenset([str(x) for x in choices])
    return AGE_MERGE.get(s, choices)

def success_rate(node_dict):
    """计算活产率"""
    total = sum(node_dict.values())
    success = node_dict.get(1, 0)
    rate = 100 * success / total if total > 0 else 0
    return f"{rate:.1f}%"

def translate_value(value):
    """翻译分类值为英文"""
    return VALUE_MAP.get(str(value), str(value))

print("✅ Helper functions defined")

In [384]:
# 样式定义
styles = {
    "root": {"shape": "ellipse", "style": "filled,bold", "fillcolor": "#FFD166", "color": "#E07A5F", "fontname": "Helvetica-Bold"},
    "split": {"shape": "box", "style": "rounded,filled", "fillcolor": "#B5EAD7", "color": "#4E878C", "fontname": "Helvetica"},
    "leaf": {"shape": "circle", "style": "filled", "fillcolor": "#F9F9F9", "color": "#999999", "fontname": "Helvetica"},
}

# 初始化图
dot = Digraph('CHAID_Frozen_LiveBirth', format='pdf')
dot.attr(rankdir='TB', splines='curved', bgcolor='white', sep='0.3', nodesep='0.3', ranksep='0.8')
dot.attr('node', fontsize='8', width='0.6', height='0.4')
dot.attr('edge', fontsize='8')

# 节点映射
node_map = {n.node_id: n for n in tree.tree_store}
children = {}
for n in tree.tree_store:
    if n.parent is not None:
        children.setdefault(n.parent, []).append(n.node_id)

print("✅ 节点映射完成")

✅ 节点映射完成


In [ ]:
# 添加节点（使用英文变量名）
for node_id, node in node_map.items():
    is_root = node.parent is None
    is_leaf = not node.split.valid()
    
    if is_leaf:
        label = success_rate(node.members)
        dot.node(f"n{node_id}", label, **styles["leaf"])
    else:
        var = node.split.column
        var_en = VAR_NAME_MAP.get(var, var)  # 翻译为英文
        pval = node.split.p
        if is_root:
            label = f"{var_en}\\np={pval:.2e}"
            dot.node(f"n{node_id}", label, **styles["root"])
        else:
            label = f"{var_en}\\np={pval:.2e}"
            dot.node(f"n{node_id}", label, **styles["split"])

# 定义年龄组的排序顺序（从左到右：>40, 36-40, 31-35, ≤30）
age_order = {'>40': 0, '36-40': 1, '31-35': 2, '≤30': 3}

def get_age_sort_key(child_node):
    """获取年龄组的排序键"""
    choices = child_node.choices
    if isinstance(choices, (list, tuple)):
        choices = choices[0] if len(choices) > 0 else ''
    return age_order.get(str(choices), 999)

# 添加边（翻译标签为英文）
for node_id, node in node_map.items():
    child_ids = children.get(node_id, [])
    
    # 如果是Age_Group节点，按照指定顺序排序子节点
    if not node.split.valid():
        continue
    if node.split.column == "Age_Group":
        child_ids = sorted(child_ids, key=lambda cid: get_age_sort_key(node_map[cid]))
    
    for child_id in child_ids:
        child = node_map[child_id]
        # 获取标签并翻译
        if node.split.column == "Age_Group":
            label = str(merge_age_label(child.choices))
        else:
            label = translate_value(child.choices)
        dot.edge(f"n{node_id}", f"n{child_id}", label=label, color="#4E878C")

# 输出
output_path = dot.render('CHAID_Frozen_LiveBirth_Tree', view=False)
print(f"✅ PDF saved to: {output_path}")